In [1]:
import sys
!"{sys.executable}" -m pip install -q pandas numpy scipy scikit-learn xgboost lifelines matplotlib seaborn requests tqdm pyarrow scikit-fuzzy


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path

ROOT = Path.cwd()
DIRS = [
    "data/raw",          # Codabench downloads
    "data/features",     # orthogonal fuzzy inputs
    "data/baselines",    # reproduced FinBoost / AutoFinSurv outputs
    "data/models",       # trained XGBoost+Cox checkpoints
    "data/fuzzy",        # IT2FIS memberships
    "data/game",         # game-theoretic simulation outputs
    "data/eval",         # bootstrap results, leaderboard submissions
    "models",
    "figures",
    "src",
]
for d in DIRS:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
print("folders ready:")
for d in DIRS:
    print(f"  {d}")

folders ready:
  data/raw
  data/features
  data/baselines
  data/models
  data/fuzzy
  data/game
  data/eval
  models
  figures
  src


In [3]:
import numpy as np

SEED = 42
np.random.seed(SEED)

# FinSurvival task definitions (16 index-outcome pairs)
INDEX_EVENTS   = ["Borrow", "Deposit", "Repay", "Withdraw"]
OUTCOME_EVENTS = ["Borrow", "Deposit", "Repay", "Withdraw", "Liquidation"]

# Train/test split per challenge spec
TRAIN_START = "2022-03-12"
TRAIN_END   = "2024-07-02"
TEST_START  = "2024-07-03"
TEST_END    = "2025-08-21"

# Fuzzy layer config
N_REGIMES  = 4
REGIME_NAMES = ["Safe", "Watch", "Danger", "Critical"]
FOU_WIDTH  = 0.20
FCM_M      = 2.0

# Bootstrap
N_BOOT     = 1000
BLOCK_SIZE = 10

print("config loaded")
print(f"  task window: {TRAIN_START} → {TEST_END}")
print(f"  regimes: {REGIME_NAMES}")

config loaded
  task window: 2022-03-12 → 2025-08-21
  regimes: ['Safe', 'Watch', 'Danger', 'Critical']


In [4]:
import pandas as pd, numpy as np, scipy, sklearn, xgboost, lifelines
import matplotlib, seaborn, requests, tqdm, pyarrow
print("core imports OK")
print(f"  pandas {pd.__version__}  numpy {np.__version__}  xgboost {xgboost.__version__}  lifelines {lifelines.__version__}")

# Verify Codabench is reachable
try:
    r = requests.get("https://www.codabench.org/", timeout=10)
    print(f"  Codabench reachable: HTTP {r.status_code}")
except Exception as e:
    print(f"  Codabench unreachable: {e}")

core imports OK
  pandas 2.3.3  numpy 2.4.6  xgboost 3.3.0  lifelines 0.30.3
  Codabench reachable: HTTP 200
